In [1]:
pip install monai loguru kagglehub scikit-learn

In [2]:
import os
import kagglehub

DATA_DIR = kagglehub.dataset_download("briscdataset/brisc2025")
print(f"Dataset downloaded to: {DATA_DIR}")

Using Colab cache for faster access to the 'brisc2025' dataset.
Dataset downloaded to: /kaggle/input/brisc2025


In [ ]:
import glob
import time
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from loguru import logger
from monai.networks.nets import AttentionUnet
from monai.networks.layers import Norm
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.utils import one_hot
from monai.transforms import (
    Compose,
    RandAffined,
    RandAdjustContrastd,
    RandFlipd,
    EnsureTyped,
    CastToTyped,
    NormalizeIntensityd,
)

# Configuration
CONFIG = {
    "data_dir": DATA_DIR,
    "batch_size": 16,
    "image_size": (512, 512),
    "num_epochs_seg": 100,
    "learning_rate_seg": 1e-4,
    "val_split": 0.2,
    "seg_channels": (32, 64, 128, 256, 512, 1024),
    "seg_strides": (2, 2, 2, 2, 2),
    "seg_in_channels": 3,
    "seg_out_channels": 2,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


print(f"Device: {CONFIG['device']}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda
GPU: NVIDIA H100 80GB HBM3
VRAM: 79.2 GB


In [4]:
class BRISCSegDataset(Dataset):
    """
    Updated PyTorch Dataset for BRISC 2025 segmentation task.
    Supports direct path injection to allow cleaner train/test splits.
    """
    def __init__(self, image_dir=None, mask_dir=None, image_size=(512, 512), augment=False):
        self.image_size = image_size
        self.augment = augment
        self.pairs = []

        # Only search for files if directories are provided
        if image_dir is not None and mask_dir is not None:
            image_path_obj = Path(image_dir)
            mask_path_obj = Path(mask_dir)
            for img_path in sorted(image_path_obj.glob("*.jpg")):
                mask_path = mask_path_obj / (img_path.stem + ".png")
                if mask_path.exists():
                    self.pairs.append((str(img_path), str(mask_path)))

        self.aug_transform = Compose([
            NormalizeIntensityd(keys=["image"], channel_wise=True),
            RandAdjustContrastd(keys=["image"], prob=0.3, gamma=(0.7, 1.3)),
            RandAffined(
                keys=["image", "label"],
                prob=0.5,
                rotate_range=(0.26, 0.26),
                scale_range=(0.1, 0.1),
                mode=("bilinear", "nearest"),
                padding_mode="zeros"
            ),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            CastToTyped(keys=["label"], dtype=np.int64),
            EnsureTyped(keys=["image", "label"])
        ])

        self.val_transform = Compose([
            NormalizeIntensityd(keys=["image"], channel_wise=True),
            CastToTyped(keys=["label"], dtype=np.int64),
            EnsureTyped(keys=["image", "label"])
        ])

    def __len__(self):
        return len(self.pairs)

    def _load_image(self, path):
        img = Image.open(path).convert("RGB")
        img = img.resize(self.image_size, Image.BILINEAR)
        return np.array(img, dtype=np.float32).transpose(2, 0, 1)

    def _load_mask(self, path):
        mask = Image.open(path).convert("L")
        mask = mask.resize(self.image_size, Image.NEAREST)
        arr = np.array(mask)
        return (arr > 127).astype(np.float32)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        data_dict = {
            "image": self._load_image(img_path),
            "label": self._load_mask(mask_path)[None, ...]
        }

        if self.augment:
            data_dict = self.aug_transform(data_dict)
        else:
            data_dict = self.val_transform(data_dict)

        return data_dict["image"], data_dict["label"]

In [5]:
from sklearn.model_selection import train_test_split

def discover_segmentation_data(data_dir):
    data_path = Path(data_dir)
    seg_dirs = list(data_path.rglob("segmentation_task"))
    if not seg_dirs:
        raise FileNotFoundError(f"Could not find 'segmentation_task' in {data_dir}")
    seg_root = seg_dirs[0]
    train_img = seg_root / "train" / "images"
    train_mask = seg_root / "train" / "masks"
    test_img = seg_root / "test" / "images"
    test_mask = seg_root / "test" / "masks"
    return str(train_img), str(train_mask), str(test_img), str(test_mask)

train_img_dir, train_mask_dir, test_img_dir, test_mask_dir = discover_segmentation_data(CONFIG["data_dir"])

# 1. Collect all valid image/mask pairs
image_path_obj = Path(train_img_dir)
mask_path_obj = Path(train_mask_dir)
all_pairs = []
for img_path in sorted(image_path_obj.glob("*.jpg")):
    mask_path = mask_path_obj / (img_path.stem + ".png")
    if mask_path.exists():
        all_pairs.append((str(img_path), str(mask_path)))

# 2. Split file paths directly using sklearn
train_pairs, val_pairs = train_test_split(
    all_pairs,
    test_size=CONFIG["val_split"],
    random_state=42,
    shuffle=True
)

# 3. Instantiate datasets and inject split pairs
train_seg = BRISCSegDataset(image_size=CONFIG["image_size"], augment=True)
train_seg.pairs = train_pairs

val_seg = BRISCSegDataset(image_size=CONFIG["image_size"], augment=False)
val_seg.pairs = val_pairs

print(f"Segmentation split: {len(train_pairs)} train / {len(val_pairs)} val")

# 4. Define Dataloaders
train_loader = DataLoader(
    train_seg,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_seg,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

Segmentation split: 3146 train / 787 val


In [6]:
def train_segmentation():
    """Train 2D U-Net on BRISC 2025 segmentation task."""
    device = CONFIG["device"]
    print(f"\n{'='*60}")
    print(" TRAINING 2D U-NET SEGMENTATION MODEL")
    print(f"{'='*60}")

    # Model
    model = AttentionUnet(
        spatial_dims=2,
        in_channels=CONFIG["seg_in_channels"],
        out_channels=CONFIG["seg_out_channels"],
        channels=CONFIG["seg_channels"],
        strides=CONFIG["seg_strides"],
        dropout=0.3,
    ).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Loss, optimizer
    class_weights = torch.tensor([1.0, 5.0]).to(device)

    loss_fn = DiceCELoss(
        to_onehot_y=True,
        softmax=True,
        squared_pred=True,
        lambda_dice=0.7,
        lambda_ce=0.3,
        weight=class_weights,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["learning_rate_seg"]
    )

    # Learning Rate Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=4,
    )

    dice_metric = DiceMetric(include_background=False, reduction="mean")
    scaler = torch.amp.GradScaler("cuda")

    best_dice = 0.0
    no_improve = 0
    train_losses, val_dices = [], []

    for epoch in range(CONFIG["num_epochs_seg"]):
        model.train()
        epoch_loss, steps = 0, 0

        for images, masks in train_loader:
            steps += 1
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                preds = model(images)
                loss = loss_fn(preds, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        epoch_loss /= max(steps, 1)
        train_losses.append(epoch_loss)

        # --- Validate every 5 epochs ---
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                for val_imgs, val_masks in val_loader:
                    val_imgs = val_imgs.to(device)
                    val_masks = val_masks.to(device).long()

                    with torch.amp.autocast("cuda"):
                        val_preds = model(val_imgs)
                        val_preds = torch.softmax(val_preds, dim=1)

                    pred_oh = one_hot(torch.argmax(val_preds, dim=1, keepdim=True), CONFIG["seg_out_channels"])
                    mask_oh = one_hot(val_masks, CONFIG["seg_out_channels"])
                    dice_metric(pred_oh, mask_oh)

            mean_dice = dice_metric.aggregate().item()
            dice_metric.reset()
            val_dices.append(mean_dice)

            # Step scheduler based on Dice
            scheduler.step(mean_dice)
            current_lr = optimizer.param_groups[0]['lr']

            print(f"Epoch {epoch+1:3d}/{CONFIG['num_epochs_seg']} | Loss: {epoch_loss:.4f} | Dice: {mean_dice:.4f}")

            if mean_dice > best_dice:
                best_dice = mean_dice
                no_improve = 0
                torch.save(model.state_dict(), "segmentation_unet.pth")
                print(f"→ Best model saved! Dice: {best_dice:.4f}")
            else:
                no_improve += 5

            if no_improve >= CONFIG["early_stop_patience"]:
                print(f"Early stopping (no improvement for {no_improve} epochs)")
                break
        else:
            print(f"Epoch {epoch+1:3d}/{CONFIG['num_epochs_seg']} | Loss: {epoch_loss:.4f}")

    print(f"\nTraining complete. Best Dice: {best_dice:.4f}")
    return model, train_losses, val_dices

In [7]:
# Train segmentation
seg_model, train_losses, val_dices = train_segmentation()


 TRAINING 2D U-NET SEGMENTATION MODEL
Model parameters: 31,805,446
Epoch   1/100 | Loss: 1.6841
Epoch   2/100 | Loss: 1.4307
Epoch   3/100 | Loss: 1.2606
Epoch   4/100 | Loss: 1.1014
Epoch   5/100 | Loss: 0.9456 | Dice: 0.6916
→ Best model saved! Dice: 0.6916
Epoch   6/100 | Loss: 0.8117
Epoch   7/100 | Loss: 0.6984
Epoch   8/100 | Loss: 0.6193
Epoch   9/100 | Loss: 0.5442
Epoch  10/100 | Loss: 0.4879 | Dice: 0.7812
→ Best model saved! Dice: 0.7812
Epoch  11/100 | Loss: 0.4482
Epoch  12/100 | Loss: 0.4135
Epoch  13/100 | Loss: 0.3849
Epoch  14/100 | Loss: 0.3613
Epoch  15/100 | Loss: 0.3432 | Dice: 0.7931
→ Best model saved! Dice: 0.7931
Epoch  16/100 | Loss: 0.3247
Epoch  17/100 | Loss: 0.3091
Epoch  18/100 | Loss: 0.2955
Epoch  19/100 | Loss: 0.2817
Epoch  20/100 | Loss: 0.2768 | Dice: 0.8326
→ Best model saved! Dice: 0.8326
Epoch  21/100 | Loss: 0.2697
Epoch  22/100 | Loss: 0.2643
Epoch  23/100 | Loss: 0.2562
Epoch  24/100 | Loss: 0.2397
Epoch  25/100 | Loss: 0.2409 | Dice: 0.8442
